Week 6 Assignment - Spark Intro

Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.
Answer

1. Driver

The Driver is the main process of a Spark application.
It creates the SparkSession/SparkContext.
It converts user code into a logical execution plan (DAG).
It schedules tasks and coordinates their execution.
It collects results from the executors.

2. Cluster Manager

The Cluster Manager is responsible for allocating resources (CPU and memory) to Spark applications.
It launches executors on worker nodes.
Examples include Standalone Cluster Manager, Hadoop YARN, Kubernetes, and Apache Mesos.

3. Executor

Executors are worker processes that run the tasks assigned by the Driver.
They perform computations on data partitions.
They store cached data in memory when needed.
They return the results of task execution back to the Driver.

Flow of Execution

User Program
      │
      ▼
   Driver
      │
      ▼
Cluster Manager
      │
      ▼
Executors
      │
      ▼
Process Data & Return Results



Q2. How does Spark's Lazy Evaluation strategy improve performance when chain-processing large datasets?

Spark's Lazy Evaluation means that transformations such as filter(), select(), and map() are not executed immediately. Instead, Spark records these operations in a Directed Acyclic Graph (DAG). Execution begins only when an action such as show(), collect(), or count() is called.

This improves performance because Spark can:

Combine multiple transformations into a single execution plan.
Eliminate unnecessary computations.
Optimize the execution using the Catalyst Optimizer.
Reduce disk I/O and network communication.

As a result, Spark processes large datasets more efficiently and with better resource utilization.

In [24]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("week_6").getOrCreate()
print("Created")

Created


In [25]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [26]:
#Q3. Write a Spark command to read a CSV file located at "data/source.csv", ensuring the first row is treated as a header and inferSchema is enabled.

df = spark.read.csv(
    "dataset.csv",
    header = True,
    inferSchema = True
)
df.show()

+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+
|product_id|  price|   category|old_name|user_id|    status| amount|base_price|region|priority|
+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+
|      1001|2188.07|Electronics|  Item_1| 5001.0| Completed|7572.98|   2188.07| South|     Low|
|      1002| 3745.5|       Home|  Item_2| 5002.0|   Pending|2626.15|    3745.5|  East|     Low|
|      1003|3001.96|     Sports|  Item_3| 5003.0| Cancelled|3398.01|   3001.96| North|  Medium|
|      1004|3607.74|      Books|  Item_4| 5004.0| Cancelled|3582.15|   3607.74| North|  Medium|
|      1005| 743.67|   Clothing|  Item_5| 5005.0| Cancelled|5223.82|    743.67| North|  Medium|
|      1006|1920.33|   Clothing|  Item_6| 5006.0|   Pending|3853.44|   1920.33|  East|  Medium|
|      1007|2342.09|       Home|  Item_7| 5007.0| Cancelled| 283.55|   2342.09|  East|  Medium|
|      1008|2592.31|     Sports|  Item_8

Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?
| CSV                  | Parquet                              |
| -------------------- | ------------------------------------ |
| Row-based storage    | Columnar storage                     |
| Plain text format    | Binary format                        |
| Larger file size     | Smaller file size due to compression |
| Slower to read       | Faster to read                       |
| No schema support    | Stores schema information            |
| Reads the entire row | Reads only required columns          |


Why it matters for performance:
Since Parquet stores data column-wise, Spark reads only the required columns instead of the entire dataset. This reduces disk I/O, memory usage, and execution time, making Parquet much faster for analytical queries.

In [27]:
#Q5. Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.

result = df.filter(col("category") == "Electronics").select("product_id", "price")

result.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      1001|2188.07|
|      1009| 1968.3|
|      1013|1739.78|
|      1014| 753.32|
|      1015| 206.96|
|      1016|1013.38|
|      1017|3955.84|
|      1020|1840.58|
|      1033|2469.34|
|      1035|4678.84|
|      1036|3810.67|
|      1041| 799.28|
|      1042|2347.17|
|      1048| 2254.1|
+----------+-------+



In [28]:
#  Q6. Write the code to revise a DataFrame by renaming the column old_name to new_name and casting the price column from a String to a Double.

df = df.withColumnRenamed("oold_name", "new_name").withColumn("price", col("price").cast("double"))

df.show()

+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+
|product_id|  price|   category|old_name|user_id|    status| amount|base_price|region|priority|
+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+
|      1001|2188.07|Electronics|  Item_1| 5001.0| Completed|7572.98|   2188.07| South|     Low|
|      1002| 3745.5|       Home|  Item_2| 5002.0|   Pending|2626.15|    3745.5|  East|     Low|
|      1003|3001.96|     Sports|  Item_3| 5003.0| Cancelled|3398.01|   3001.96| North|  Medium|
|      1004|3607.74|      Books|  Item_4| 5004.0| Cancelled|3582.15|   3607.74| North|  Medium|
|      1005| 743.67|   Clothing|  Item_5| 5005.0| Cancelled|5223.82|    743.67| North|  Medium|
|      1006|1920.33|   Clothing|  Item_6| 5006.0|   Pending|3853.44|   1920.33|  East|  Medium|
|      1007|2342.09|       Home|  Item_7| 5007.0| Cancelled| 283.55|   2342.09|  East|  Medium|
|      1008|2592.31|     Sports|  Item_8

Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

Spark maintains a Lineage Graph (Directed Acyclic Graph) that records all transformations performed on the data. Instead of replicating intermediate data, Spark remembers how each dataset was created.

If a worker node fails and some data is lost:

Spark uses the lineage graph to identify the lost partitions.
It recomputes only the missing partitions by replaying the required transformations.
Other completed partitions remain unchanged.

This provides fault tolerance without requiring redundant copies of intermediate data, making Spark both reliable and memory-efficient.

In [29]:
# Q8. Write a query to filter a DataFrame df_orders for rows where the status is 'Completed' AND the amount is greater than 1000.

result = df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

result.show()

+----------+-------+-----------+--------+-------+---------+-------+----------+------+--------+
|product_id|  price|   category|old_name|user_id|   status| amount|base_price|region|priority|
+----------+-------+-----------+--------+-------+---------+-------+----------+------+--------+
|      1001|2188.07|Electronics|  Item_1| 5001.0|Completed|7572.98|   2188.07| South|     Low|
|      1012| 609.91|     Sports| Item_12| 5012.0|Completed|7172.08|    609.91|  East|     Low|
|      1013|1739.78|Electronics| Item_13|   NULL|Completed|3180.38|   1739.78| South|    High|
|      1018| 493.46|   Clothing| Item_18| 5018.0|Completed| 3125.2|    493.46|  West|    High|
|      1020|1840.58|Electronics| Item_20| 5020.0|Completed|4269.07|   1840.58|  East|     Low|
|      1024|4669.09|       Home| Item_24| 5024.0|Completed|6137.78|   4669.09| North|  Medium|
|      1035|4678.84|Electronics| Item_35| 5035.0|Completed|3754.49|   4678.84|  East|     Low|
|      1036|3810.67|Electronics| Item_36| 5036.0|C

Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

Predicate Pushdown is an optimization technique in which Spark pushes filter conditions down to the Parquet storage layer before reading the data.

Instead of loading the entire dataset into memory, Parquet reads only the rows that satisfy the filter condition.

Benefits:

Reduces disk I/O.
Loads less data into memory.
Improves query performance.
Speeds up filtering operations on large datasets.

For example, if a query filters age > 30, Parquet skips row groups that cannot satisfy this condition, reducing the amount of data read.

In [30]:
# Q10. Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

df = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

df.show()

+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+------------------+
|product_id|  price|   category|old_name|user_id|    status| amount|base_price|region|priority|       final_price|
+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+------------------+
|      1001|2188.07|Electronics|  Item_1| 5001.0| Completed|7572.98|   2188.07| South|     Low|         2581.9226|
|      1002| 3745.5|       Home|  Item_2| 5002.0|   Pending|2626.15|    3745.5|  East|     Low|           4419.69|
|      1003|3001.96|     Sports|  Item_3| 5003.0| Cancelled|3398.01|   3001.96| North|  Medium|3542.3127999999997|
|      1004|3607.74|      Books|  Item_4| 5004.0| Cancelled|3582.15|   3607.74| North|  Medium| 4257.133199999999|
|      1005| 743.67|   Clothing|  Item_5| 5005.0| Cancelled|5223.82|    743.67| North|  Medium| 877.5305999999999|
|      1006|1920.33|   Clothing|  Item_6| 5006.0|   Pending|3853.44|   1920.33| 

Q11. What is the difference between Transformations and Actions? Provide two examples of each.

| Transformations                     | Actions                              |
| ----------------------------------- | ------------------------------------ |
| Create a new DataFrame or RDD       | Trigger execution of transformations |
| Evaluated lazily                    | Executed immediately                 |
| Do not produce output by themselves | Produce results or write data        |

Examples of Transformations:

filter()
select()

Examples of Actions:

show()
count()

Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

| Client Mode                                   | Cluster Mode                               |
| --------------------------------------------- | ------------------------------------------ |
| Driver runs on the client machine             | Driver runs inside the cluster             |
| Client must remain connected during execution | Client can disconnect after job submission |
| Suitable for development and testing          | Suitable for production environments       |
| Driver uses client machine resources          | Driver uses cluster resources              |


In [38]:
# Q14. Write a query to filter a dataset for rows where the region is 'North' OR the priority is 'High'.

result = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

result.show()

+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+
|product_id|  price|   category|old_name|user_id|    status| amount|base_price|region|priority|
+----------+-------+-----------+--------+-------+----------+-------+----------+------+--------+
|      1003|3001.96|     Sports|  Item_3| 5003.0| Cancelled|3398.01|   3001.96| North|  Medium|
|      1004|3607.74|      Books|  Item_4| 5004.0| Cancelled|3582.15|   3607.74| North|  Medium|
|      1005| 743.67|   Clothing|  Item_5| 5005.0| Cancelled|5223.82|    743.67| North|  Medium|
|      1008|2592.31|     Sports|  Item_8| 5008.0| Cancelled|3079.17|   2592.31| North|     Low|
|      1010|4929.58|     Sports| Item_10| 5010.0| Cancelled|1662.02|   4929.58| North|     Low|
|      1013|1739.78|Electronics| Item_13|   NULL| Completed|3180.38|   1739.78| South|    High|
|      1015| 206.96|Electronics| Item_15| 5015.0|   Pending| 498.19|    206.96| North|    High|
|      1018| 493.46|   Clothing| Item_18

Q15. When exploring a dataset, why is it safer to use .show(5) instead of .collect() on a multi-terabyte dataset?

The .show(5) function displays only the first five rows of a DataFrame, while .collect() retrieves all rows and transfers them to the Driver's memory.

On a multi-terabyte dataset, using .collect() can:

Consume excessive Driver memory.
Cause an OutOfMemoryError.
Increase network traffic and execution time.
Potentially crash the application.

In contrast, .show(5) reads and displays only a small sample of the data, making it safe and efficient for inspecting large datasets.